# Project 1: Medical Disease Prediction - Level 1
## Building Your First Diabetes Prediction ML Model

Welcome! By the end of this notebook, you'll have built a working ML pipeline that predicts diabetes risk from patient health data.

**What you'll learn:**
- How to load and explore real medical data
- Data preprocessing and handling missing values
- Building and training a machine learning model
- Evaluating model performance
- Making predictions on new patient data

**Time estimate:** 30-45 minutes

---

## Step 1: Import Required Libraries

These are the essential Python libraries we'll use:
- **pandas**: For working with data (loading, exploring, cleaning)
- **numpy**: For numerical computations
- **scikit-learn**: ML algorithms and tools
- **matplotlib & seaborn**: For data visualization

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# For nicer visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")

## Step 2: Load and Explore Your Data

First, let's load the Diabetes dataset and understand what we're working with.

**What each column means:**
- **Pregnancies**: Number of times pregnant
- **Glucose**: Plasma glucose concentration (higher = more sugar in blood)
- **BloodPressure**: Diastolic blood pressure
- **SkinThickness**: Triceps skin fold thickness (indicator of body fat)
- **Insulin**: 2-Hour serum insulin level
- **BMI**: Body Mass Index (weight indicator)
- **DiabetesPedigreeFunction**: Genetic risk factor
- **Age**: Patient's age in years
- **Outcome**: YES = has diabetes, NO = doesn't have diabetes (🎯 **This is what we're predicting!**)

In [ ]:
# Load the dataset
df = pd.read_csv('dataset/Diabetes.csv')

# Display first few rows
print("First 5 rows of data:")
print(df.head())
print("\n" + "="*60)

# Dataset shape and info
print(f"\nDataset shape: {df.shape[0]} patients, {df.shape[1]} features")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nBasic statistics:")
print(df.describe())

## Step 3: Data Preprocessing & Handling Missing Values

In real medical data, some values might be missing or invalid (like blood pressure = 0). We need to handle these before training our model.

In [ ]:
# Check for missing values
print("Missing values (NaN):")
print(df.isnull().sum())

# Some medical readings of 0 are actually missing data (e.g., blood pressure can't be 0)
# Let's identify and handle these
columns_with_zero_issues = ['Diastolic blood pressure', 'Triceps skin fold thickness', 
                             '2-Hour serum insulin', 'Body mass index']

print("\nColumns with potentially invalid zero values:")
for col in columns_with_zero_issues:
    zero_count = (df[col] == 0).sum()
    print(f"  {col}: {zero_count} zeros")

# Replace zeros with the mean value of that column
# (This is a common preprocessing technique)
for col in columns_with_zero_issues:
    mean_val = df[df[col] != 0][col].mean()
    df[col].replace(0, mean_val, inplace=True)

print("\n✓ Data cleaned! Zeros replaced with column means.")
print("\nData after preprocessing:")
print(df.head())

## Step 4: Prepare Data for Machine Learning

We need to:
1. **Separate features (X) from target (y)**: X = patient health data, y = whether they have diabetes
2. **Split into training and testing**: Train on 80% of data, test on 20% (unseen data)
3. **Scale the features**: Make all numbers comparable (important for ML algorithms)

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Class variable', axis=1)  # All columns except the target
y = df['Class variable']  # Only the target column

# Convert 'YES'/'NO' to 1/0 for the model
y = (y == 'YES').astype(int)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nTarget distribution:")
print(f"  NO diabetes (0): {(y == 0).sum()} patients")
print(f"  YES diabetes (1): {(y == 1).sum()} patients")

# Split into training (80%) and testing (20%) data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set: {X_train.shape[0]} patients")
print(f"Testing set: {X_test.shape[0]} patients")

## Step 5: Scale Features & Build the Model

**Feature Scaling**: Algorithms work better when features are on similar scales (e.g., 0-1). This is called normalization.

**Logistic Regression**: We're using this algorithm because:
- ✅ Great for beginners (easy to understand)
- ✅ Fast to train
- ✅ Works well for binary classification (YES/NO problems)
- ✅ Gives probability scores (e.g., "70% chance of diabetes")

In [ ]:
# Initialize the scaler and fit it on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit AND transform
X_test_scaled = scaler.transform(X_test)        # Only transform

print("✓ Features scaled!")
print(f"\nOriginal feature range (sample): {X_train.iloc[0].min():.1f} to {X_train.iloc[0].max():.1f}")
print(f"Scaled feature range (sample): {X_train_scaled[0].min():.2f} to {X_train_scaled[0].max():.2f}")

# Initialize and train the Logistic Regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

print("\n" + "="*60)
print("✓ MODEL TRAINED SUCCESSFULLY!")
print("="*60)

## Step 6: Evaluate Your Model

Let's see how well our model performs on new (test) data it hasn't seen before.

**Key metrics:**
- **Accuracy**: % of correct predictions
- **Confusion Matrix**: Shows True Positives, False Positives, etc.
- **Classification Report**: Precision, Recall, F1-Score

In [ ]:
# Make predictions on test data
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")
print(f"\nThis means the model correctly predicts diabetes in {accuracy:.1%} of cases!")

# Confusion Matrix
print("\n" + "="*60)
print("CONFUSION MATRIX:")
print("="*60)
cm = confusion_matrix(y_test, y_pred)
print(f"\nTrue Negatives (Correctly said NO):  {cm[0,0]}")
print(f"False Positives (Wrongly said YES):   {cm[0,1]}")
print(f"False Negatives (Wrongly said NO):    {cm[1,0]}")
print(f"True Positives (Correctly said YES):  {cm[1,1]}")

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Detailed classification report
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

## Step 7: Make Predictions on New Patient Data 🎯

This is the most important part! Let's use our trained model to predict diabetes risk for a new patient.

In [ ]:
# Example: New patient data (single patient)
# Order: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
new_patient = [[2, 120, 70, 25, 100, 28.5, 0.5, 35]]

# Scale the new patient's data using our trained scaler
new_patient_scaled = scaler.transform(new_patient)

# Make prediction
prediction = model.predict(new_patient_scaled)[0]  # 0 = No, 1 = Yes
probability = model.predict_proba(new_patient_scaled)[0]

print("="*60)
print("NEW PATIENT PREDICTION")
print("="*60)
print(f"\nPatient Characteristics:")
print(f"  Pregnancies: {new_patient[0][0]}")
print(f"  Glucose: {new_patient[0][1]} mg/dL")
print(f"  Blood Pressure: {new_patient[0][2]} mmHg")
print(f"  BMI: {new_patient[0][5]}")
print(f"  Age: {new_patient[0][7]} years")

print(f"\n" + "="*60)
print(f"PREDICTION RESULT:")
print(f"="*60)
if prediction == 0:
    print(f"❌ NO DIABETES PREDICTED")
    print(f"   Confidence: {probability[0]:.1%}")
else:
    print(f"⚠️ DIABETES PREDICTED")
    print(f"   Confidence: {probability[1]:.1%}")
    
print(f"\nDetailed Probabilities:")
print(f"  No Diabetes: {probability[0]:.1%}")
print(f"  Diabetes: {probability[1]:.1%}")

## Try it Yourself! 🚀

Modify the `new_patient` data above with different values and see how the prediction changes. Try:
- A younger patient
- A patient with higher glucose
- A patient with higher BMI

**What did you notice?** Which factors seem to increase diabetes risk?

## 🎓 What You Just Learned!

### Core ML Concepts:
1. **Data Loading & Exploration**: Understanding your dataset before modeling
2. **Data Preprocessing**: Cleaning and handling missing/invalid data
3. **Feature Scaling**: Normalizing features for better model performance
4. **Train-Test Split**: Evaluating on unseen data to avoid overfitting
5. **Logistic Regression**: A fundamental classification algorithm
6. **Model Evaluation**: Accuracy, Confusion Matrix, Classification Report
7. **Prediction**: Using your model on new data

### Real-World Applications:
- 🏥 Healthcare: Risk prediction and early diagnosis
- 💊 Pharma: Patient stratification and clinical trials
- 📊 Insurance: Risk assessment

---

## 🚀 Next Steps (Level 2):
- Build a **Streamlit web app** to make predictions interactive
- Add **SHAP explainability** to understand why the model makes predictions
- Try **advanced models** (Random Forest, XGBoost)
- Deploy your model as an **API**

**Ready for Level 2?** Let me know!